# MMA3001 Individual Project
## Predictive Maintenance using the AI4I 2020 Dataset

### Stage 2 — Baseline Modelling and Validation Setup

Stage 1 established that the dataset is strongly imbalanced, with only 3.39% of observations corresponding to machine failure.

This stage establishes a reproducible validation procedure and a simple baseline classifier before more advanced predictive methods are investigated.

The objectives are to:

- define the predictor and target variables selected in Stage 1;
- create a reproducible stratified training/test split;
- verify that the target distribution is preserved;
- establish a majority-class baseline;
- evaluate the baseline using metrics suitable for an imbalanced failure-detection problem; and
- define a consistent evaluation framework for later model comparison.

No model optimisation is performed during this stage.

In [ ]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

RANDOM_STATE = 42

# ---------------------------------------------------------
# Model variables selected during Stage 1
# ---------------------------------------------------------

TARGET = "Machine failure"

FEATURES = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

# ---------------------------------------------------------
# Locate project dataset
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd()

# Allow the notebook to run from either the repository root
# or the /notebooks directory
if not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}"
    )
df = pd.read_csv(DATA_PATH)
print(
    f"Dataset loaded: "
    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
)

Dataset loaded: 10,000 rows × 14 columns


In [ ]:
# ---------------------------------------------------------
# Verify required modelling variables
# ---------------------------------------------------------
required_columns = FEATURES + [TARGET]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values detected in modelling variables."
    )

# ---------------------------------------------------------
# Define predictors (X) and target (y)
# ---------------------------------------------------------
X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f"Predictor matrix shape: {X.shape}")
print(f"Target vector shape:    {y.shape}")

print("\nPredictor variables:")
for feature in FEATURES:
    print(f"  - {feature}")

print(f"\nTarget variable:")
print(f"  - {TARGET}")

Predictor matrix shape: (10000, 6)
Target vector shape:    (10000,)

Predictor variables:
  - Type
  - Air temperature [K]
  - Process temperature [K]
  - Rotational speed [rpm]
  - Torque [Nm]
  - Tool wear [min]

Target variable:
  - Machine failure


In [ ]:
# ---------------------------------------------------------
# Reproducible stratified split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
# ---------------------------------------------------------
# Verify split
# ---------------------------------------------------------
split_summary = pd.DataFrame({
    "Observations": [
        len(y_train),
        len(y_test),
    ],
    "Failures": [
        int(y_train.sum()),
        int(y_test.sum()),
    ],
    "Failure Rate (%)": [
        100 * y_train.mean(),
        100 * y_test.mean(),
    ],
}, index=["Training", "Test"])

display(split_summary.round(2))

,Observations,Failures,Failure Rate (%)
Training,8000,271,3.39
Test,2000,68,3.40


In [ ]:
# ---------------------------------------------------------
# Train majority-class baseline
# ---------------------------------------------------------
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

baseline_model.fit(
    X_train,
    y_train,
)

y_pred_baseline = baseline_model.predict(
    X_test
)
# ---------------------------------------------------------
# Calculate evaluation metrics
# ---------------------------------------------------------
baseline_metrics = pd.DataFrame({
    "Score": {
        "Accuracy": accuracy_score(
            y_test,
            y_pred_baseline,
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test,
            y_pred_baseline,
        ),
        "Precision": precision_score(
            y_test,
            y_pred_baseline,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test,
            y_pred_baseline,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_test,
            y_pred_baseline,
            zero_division=0,
        ),
    }
})

print("Majority-class baseline performance:")
display(baseline_metrics.round(4))
# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------
cm = confusion_matrix(
    y_test,
    y_pred_baseline,
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual normal",
        "Actual failure",
    ],
    columns=[
        "Predicted normal",
        "Predicted failure",
    ],
)

print("\nConfusion matrix:")
display(cm_df)

Majority-class baseline performance:


,Score
Accuracy,0.966
Balanced Accuracy,0.500
Precision,0.000
Recall,0.000
F1-score,0.000



Confusion matrix:


,Predicted normal,Predicted failure
Actual normal,1932,0
Actual failure,68,0
